Here are functions to exort for model building

In [41]:
#| default_exp anthro_module

In [42]:
#| export
import pickle
import pandas as pd
import arviz as az
import bambi as bmb

In [43]:
#| export
def make_pickle(model, kindofmodel, measurement):
    with open(f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}','wb') as file:
        pickle.dump(model,file)

In [44]:
#| export
def measurement_names():
    test=pd.read_csv('../data/processed/ANSURIItest.csv')
    test_var=test.iloc[:,1:94].drop('weightkg',axis=1).drop('stature',axis=1)
    col = test_var.columns[:]
    return col

### predict with bambi, mean values

In [45]:
#| export
def model_bmb(measurement):
    train=pd.read_csv('../data/processed/ANSURIInormalizedtrain.csv')
    variables = f"{measurement} ~ weightkg + stature + Gender"
    made_model = bmb.Model(variables, data=train)
    return made_model

In [46]:
#| export
def make_model_formula(measurement, formula:str):
    train=pd.read_csv('../data/processed/ANSURIIminusmeanstrain.csv')
    variables = f"{measurement} ~ " + formula
    made_model = bmb.Model(variables,data=train)
    return made_model

In [47]:
#| export
def fit_model(measurement, formula:str):
    made_model = make_model_formula(measurement, formula)
    made_fitted = made_model.fit(tune=2000, draws=2000, chains = 4, cores= 1, init="adapt_diag", progressbar=True)
    return made_fitted

In [48]:
#| export
def predict_mean_bmb(fitted, model, data, measurement):
    pred_df=pd.DataFrame()
    model.predict(fitted, data = data, kind='response')
    posterior_predictive = az.extract(fitted, group="posterior_predictive")
    predictions = posterior_predictive[measurement]
    mean_pred=[]
    for row in predictions:
        mean_pred.append(row.values.mean())    
    pred_df[measurement]=mean_pred
    return pred_df

In [40]:
import nbdev; nbdev.nbdev_export()